In [0]:
from framework.core.configuration import *

print(BRONZE_LAYER)
print(SILVER_LAYER)

In [0]:
import sys

for p in sys.path:
    print(p)

In [0]:
from framework.core.configuration import BRONZE_LAYER

print(BRONZE_LAYER)

In [0]:
display(
    spark.table("smip.bronze.machines")
)

spark.table("smip.bronze.machines").printSchema()

In [0]:
tables = [row.tableName for row in spark.sql("SHOW TABLES IN smip.bronze").collect()]

for table in tables:
    print(f"Schema for smip.bronze.{table}:")
    spark.table(f"smip.bronze.{table}").printSchema()

In [0]:
display(
    spark.sql(f"SHOW TABLES IN {SILVER_LAYER}")
)

In [0]:
from datetime import datetime, timedelta

all_tables = spark.sql("SHOW TABLES IN smip.bronze").collect()
current_time = datetime.now()
five_minutes_ago = current_time - timedelta(minutes=5)

recent_tables = []
for row in all_tables:
    max_timestamp = spark.sql(f"SELECT max(ingestion_timestamp) FROM smip.bronze.{row.tableName}").collect()[0][0]
    if max_timestamp and max_timestamp >= five_minutes_ago:
        recent_tables.append(row.tableName)

for table in recent_tables[:9]:
    print(f"Schema for smip.bronze.{table}:")
    spark.table(f"smip.bronze.{table}").printSchema()

In [0]:
from framework.core.configuration import SILVER_LAYER

print(SILVER_LAYER)

In [0]:
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT serial_number) AS distinct_serials
        FROM {SILVER_LAYER}.fact_production
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT press_operation_id) AS distinct_operations
        FROM {SILVER_LAYER}.fact_press_operations
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) AS missing_products,
            SUM(CASE WHEN machine_key IS NULL THEN 1 ELSE 0 END) AS missing_machines,
            SUM(CASE WHEN operator_key IS NULL THEN 1 ELSE 0 END) AS missing_operators,
            SUM(CASE WHEN tool_key IS NULL THEN 1 ELSE 0 END) AS missing_tools,
            SUM(CASE WHEN factory_key IS NULL THEN 1 ELSE 0 END) AS missing_factory
        FROM {SILVER_LAYER}.fact_press_operations
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            quality_result,
            COUNT(*) AS operations
        FROM {SILVER_LAYER}.fact_press_operations
        GROUP BY quality_result
        ORDER BY operations DESC
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            COUNT(*) total_rows,
            COUNT(DISTINCT test_result_id) distinct_tests
        FROM {SILVER_LAYER}.fact_quality
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            result,
            COUNT(*)
        FROM {SILVER_LAYER}.fact_quality
        GROUP BY result
    """)
)

In [0]:
from framework.core.configuration import SILVER_LAYER

display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_points,
            COUNT(DISTINCT press_operation_id) AS operations
        FROM {SILVER_LAYER}.fact_force_curve
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) AS missing_products,
            SUM(CASE WHEN machine_key IS NULL THEN 1 ELSE 0 END) AS missing_machines,
            SUM(CASE WHEN operator_key IS NULL THEN 1 ELSE 0 END) AS missing_operators,
            SUM(CASE WHEN tool_key IS NULL THEN 1 ELSE 0 END) AS missing_tools
        FROM {SILVER_LAYER}.fact_force_curve
    """)
)

In [0]:
display(
    spark.read.csv(
        "/Volumes/smip/bronze/source_data/transactional_data/work_orders.csv",
        header=True
    )
    .groupBy("planned_shift")
    .count()
)

In [0]:
from generator.configs.factory_digital_twin import ShiftType

print(list(ShiftType))

In [0]:
%restart_python

In [0]:
import inspect
from generator.engine.production_planner import ProductionPlanner

print(inspect.getsource(ProductionPlanner._generate_day))

In [0]:
%sql
DESCRIBE smip.gold.vw_production_summary;

In [0]:
%sql
DESCRIBE smip.gold.vw_quality_summary;

In [0]:
%sql
DESCRIBE smip.gold.vw_oee_summary;

In [0]:
%sql
DESCRIBE smip.gold.vw_traceability_summary;

In [0]:
%sql
DESCRIBE smip.gold.vw_executive_summary;